## Part 3.1: Create, Load, and Validate Data Quality Rules

### Objective

Create or load the persistent data-quality rules used to evaluate the contents of the Cyclistic source files.

Part 2 verified the structural schema of the source data. Part 3 evaluates whether the values inside that structure are complete, valid, consistent, and suitable for further processing.

This section defines the quality contract only. It does not scan source files or calculate quality metrics.

### Approved Schema Dependency

Data-quality rules must be associated with an approved schema contract.

Before the rules are initialised, this section verifies that:

* `active_schema_status` is `APPROVED`;
* `active_schema` is not empty;
* `active_schema_version` is populated;
* every enabled quality rule references columns available in the approved schema.

If the active schema has not been approved, the quality audit must not continue.

### Quality Rule Structure

The quality rules are stored in `quality_rules.json`.

Each rule contains:

* a unique rule ID;
* a category;
* the affected column or columns;
* a severity;
* an enabled status;
* rule-specific parameters;
* a human-readable description.

Supported severity levels are:

```text
WARNING
FAIL
```

A warning identifies a suspicious condition that does not automatically invalidate the row.

A failure identifies a violation that may cause the row to become a quarantine candidate during later processing.

### Initial Quality Contract

The initial rules cover:

* required values;
* blank identifiers;
* duplicate ride IDs;
* permitted categorical values;
* datetime completeness and ordering;
* ride-duration validity;
* missing station information;
* latitude and longitude ranges.

The initial decisions are:

```text
Missing or blank ride_id                    → FAIL
Duplicate ride_id across the dataset        → FAIL
Invalid member_casual value                 → FAIL
Missing started_at or ended_at              → FAIL
ended_at not later than started_at          → FAIL
Ride duration greater than 24 hours         → WARNING
Missing station names or IDs                → WARNING
Coordinates outside global geographic range → FAIL
```

Missing station information is treated as a warning because some ride types may not begin or end at a formal station.

### File-Management Policy

This section follows the same conservative persistence policy used by the schema profiler:

* create `quality_rules.json` when it does not exist;
* preserve an existing valid file;
* add missing top-level sections from defaults;
* add missing default rules without replacing existing customised rules;
* preserve invalid JSON files for human review;
* use in-memory defaults when an existing file cannot be parsed;
* write safe migrations atomically;
* record all creations, migrations, warnings, and failures in `pipeline_audit.log`.

### Expected Outcome

* The approved schema prerequisite is verified.
* `quality_rules.json` exists or safe in-memory defaults are available.
* Existing valid customised rules are preserved.
* Missing required rules are added safely.
* Rule IDs, severities, columns, and parameters are validated.
* Enabled rules reference columns available in the approved schema.
* `quality_rules` is ready for metric generation and assessment in later Part 3 stages.


In [ ]:
# ==========================================
# Part 3.1: Create, Load, Migrate, and
# Validate Data Quality Rules
# ==========================================

import json
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from uuid import uuid4


# ------------------------------------------
# Validate required dependencies
# ------------------------------------------

REQUIRED_PART_3_1_VARIABLES = {
    "QUALITY_RULES_PATH": QUALITY_RULES_PATH,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
}

for variable_name, variable_value in (
    REQUIRED_PART_3_1_VARIABLES.items()
):
    if variable_value is None:
        raise RuntimeError(
            f"Required Part 3.1 variable is unavailable: "
            f"{variable_name}"
        )


for path_name in (
    "QUALITY_RULES_PATH",
    "MASTER_SCHEMA_PATH",
    "PIPELINE_AUDIT_PATH",
):
    if not isinstance(
        REQUIRED_PART_3_1_VARIABLES[path_name],
        Path,
    ):
        raise TypeError(
            f"{path_name} must be a pathlib.Path object."
        )


# ------------------------------------------
# Initialise quality-audit execution state
# ------------------------------------------

QUALITY_AUDIT_SESSION_ID = uuid4().hex[:8]

QUALITY_AUDIT_START_TIME = datetime.now(
    MELBOURNE_TIMEZONE
)

dq_run_warnings = []
dq_run_errors = []


SUPPORTED_DQ_LOG_LEVELS = {
    "INFO",
    "WARNING",
    "ERROR",
    "CRITICAL",
}


def log_quality_event(
    message: str,
    level: str = "INFO",
) -> str:
    """
    Record one quality-audit event using a dedicated audit-session
    identifier.
    """
    normalised_level = level.upper().strip()

    if normalised_level not in SUPPORTED_DQ_LOG_LEVELS:
        raise ValueError(
            f"Unsupported quality log level: {level}"
        )

    if not isinstance(message, str) or not message.strip():
        raise ValueError(
            "Quality audit message must be a non-empty string."
        )

    timestamp = datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds")

    log_entry = (
        f"[{timestamp}] "
        f"[DQ_RUN:{QUALITY_AUDIT_SESSION_ID}] "
        f"[{normalised_level}] "
        f"{message.strip()}"
    )

    with PIPELINE_AUDIT_PATH.open(
        mode="a",
        encoding="utf-8",
    ) as audit_file:
        audit_file.write(log_entry + "\n")

    print(log_entry)

    event_record = {
        "timestamp": timestamp,
        "quality_audit_session_id": (
            QUALITY_AUDIT_SESSION_ID
        ),
        "level": normalised_level,
        "message": message.strip(),
    }

    if normalised_level == "WARNING":
        dq_run_warnings.append(event_record)

    elif normalised_level in {
        "ERROR",
        "CRITICAL",
    }:
        dq_run_errors.append(event_record)

    return log_entry


log_quality_event(
    message=(
        "QUALITY_AUDIT_SESSION_START: "
        "Data quality audit initialised."
    ),
    level="INFO",
)


# ------------------------------------------
# Google Drive-tolerant JSON writer
# ------------------------------------------

def save_quality_json(
    file_path: Path,
    data: dict,
) -> None:
    """
    Write JSON through a temporary file and fall back to a direct
    write if Google Drive replacement fails.
    """
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        f".{file_path.name}.tmp"
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
        ) as temporary_file:
            json.dump(
                data,
                temporary_file,
                indent=4,
                ensure_ascii=False,
            )
            temporary_file.write("\n")

        try:
            temporary_path.replace(file_path)

        except OSError as replace_error:
            log_quality_event(
                message=(
                    "QUALITY_JSON_REPLACE_FALLBACK: "
                    f"path={file_path}; "
                    f"error={replace_error}."
                ),
                level="WARNING",
            )

            with file_path.open(
                mode="w",
                encoding="utf-8",
            ) as target_file:
                json.dump(
                    data,
                    target_file,
                    indent=4,
                    ensure_ascii=False,
                )
                target_file.write("\n")

    finally:
        if temporary_path.exists():
            temporary_path.unlink()


# ------------------------------------------
# Reload and validate approved master schema
# ------------------------------------------

try:
    with MASTER_SCHEMA_PATH.open(
        mode="r",
        encoding="utf-8",
    ) as master_schema_file:
        approved_master_registry = json.load(
            master_schema_file
        )

except json.JSONDecodeError as error:
    log_quality_event(
        message=(
            "QUALITY_RULE_INITIALISATION_FAILED: "
            "master_schema.json contains invalid JSON. "
            f"Error={error}"
        ),
        level="CRITICAL",
    )

    raise ValueError(
        "master_schema.json contains invalid JSON."
    ) from error


if not isinstance(approved_master_registry, dict):
    raise TypeError(
        "master_schema.json must contain a JSON object."
    )


ACTIVE_SCHEMA = approved_master_registry.get(
    "active_schema",
    {},
)

ACTIVE_SCHEMA_VERSION = approved_master_registry.get(
    "active_schema_version"
)

ACTIVE_SCHEMA_STATUS = approved_master_registry.get(
    "active_schema_status"
)


if ACTIVE_SCHEMA_STATUS != "APPROVED":
    log_quality_event(
        message=(
            "QUALITY_RULE_INITIALISATION_BLOCKED: "
            f"active_schema_status={ACTIVE_SCHEMA_STATUS}; "
            "expected=APPROVED."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Part 3 requires an approved active schema."
    )


if not isinstance(ACTIVE_SCHEMA, dict) or not ACTIVE_SCHEMA:
    log_quality_event(
        message=(
            "QUALITY_RULE_INITIALISATION_BLOCKED: "
            "active_schema is empty or invalid."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Part 3 requires a non-empty active schema."
    )


if (
    not isinstance(ACTIVE_SCHEMA_VERSION, str)
    or not ACTIVE_SCHEMA_VERSION.strip()
):
    log_quality_event(
        message=(
            "QUALITY_RULE_INITIALISATION_BLOCKED: "
            "active_schema_version is missing."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Part 3 requires an active schema version."
    )


# ------------------------------------------
# Define default quality contract
# ------------------------------------------

quality_rules_initialisation_time = datetime.now(
    MELBOURNE_TIMEZONE
).isoformat(timespec="seconds")


DEFAULT_QUALITY_RULES = {
    "metadata": {
        "ruleset_version": "1.1.0",
        "ruleset_status": "ACTIVE",
        "source_schema_version": ACTIVE_SCHEMA_VERSION,
        "created_at": quality_rules_initialisation_time,
        "updated_at": quality_rules_initialisation_time,
        "description": (
            "Cyclistic source-data quality contract."
        ),
    },
    "rules": {
        "DQ001_RIDE_ID_REQUIRED": {
            "category": "COMPLETENESS",
            "columns": ["ride_id"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "ride_id must not be null."
            ),
        },
        "DQ002_RIDE_ID_NOT_BLANK": {
            "category": "COMPLETENESS",
            "columns": ["ride_id"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_BLANK",
            "parameters": {},
            "description": (
                "ride_id must not be empty or whitespace-only."
            ),
        },
        "DQ003_RIDE_ID_UNIQUE": {
            "category": "UNIQUENESS",
            "columns": ["ride_id"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "UNIQUE",
            "parameters": {
                "scope": "ALL_FILES",
            },
            "description": (
                "ride_id must be unique across all source files."
            ),
        },
        "DQ004_RIDEABLE_TYPE_ALLOWED": {
            "category": "VALIDITY",
            "columns": ["rideable_type"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "ALLOWED_VALUES",
            "parameters": {
                "allowed_values": [
                    "classic_bike",
                    "electric_bike",
                    "docked_bike",
                ],
                "case_sensitive": True,
            },
            "description": (
                "rideable_type must contain an approved value."
            ),
        },
        "DQ005_STARTED_AT_REQUIRED": {
            "category": "COMPLETENESS",
            "columns": ["started_at"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "started_at must not be null."
            ),
        },
        "DQ006_ENDED_AT_REQUIRED": {
            "category": "COMPLETENESS",
            "columns": ["ended_at"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "ended_at must not be null."
            ),
        },
        "DQ007_END_AFTER_START": {
            "category": "CONSISTENCY",
            "columns": [
                "started_at",
                "ended_at",
            ],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "DATETIME_ORDER",
            "parameters": {
                "operator": "GREATER_THAN",
            },
            "description": (
                "ended_at must be later than started_at."
            ),
        },
        "DQ008_DURATION_MAXIMUM": {
            "category": "REASONABLENESS",
            "columns": [
                "started_at",
                "ended_at",
            ],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "DURATION_MAXIMUM",
            "parameters": {
                "maximum_hours": 24,
            },
            "description": (
                "Ride duration greater than 24 hours requires "
                "review."
            ),
        },
        "DQ009_MEMBER_TYPE_ALLOWED": {
            "category": "VALIDITY",
            "columns": ["member_casual"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "ALLOWED_VALUES",
            "parameters": {
                "allowed_values": [
                    "member",
                    "casual",
                ],
                "case_sensitive": True,
            },
            "description": (
                "member_casual must be member or casual."
            ),
        },
        "DQ010_START_STATION_NAME_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["start_station_name"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing start station names require review."
            ),
        },
        "DQ011_START_STATION_ID_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["start_station_id"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing start station IDs require review."
            ),
        },
        "DQ012_END_STATION_NAME_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["end_station_name"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing end station names require review."
            ),
        },
        "DQ013_END_STATION_ID_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["end_station_id"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing end station IDs require review."
            ),
        },

        # Coordinate range rules now allow null values.
        # Completeness is evaluated separately by DQ018-DQ021.
        "DQ014_START_LAT_RANGE": {
            "category": "VALIDITY",
            "columns": ["start_lat"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -90,
                "maximum": 90,
                "allow_null": True,
            },
            "description": (
                "Non-null start_lat values must be between "
                "-90 and 90."
            ),
        },
        "DQ015_END_LAT_RANGE": {
            "category": "VALIDITY",
            "columns": ["end_lat"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -90,
                "maximum": 90,
                "allow_null": True,
            },
            "description": (
                "Non-null end_lat values must be between "
                "-90 and 90."
            ),
        },
        "DQ016_START_LNG_RANGE": {
            "category": "VALIDITY",
            "columns": ["start_lng"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -180,
                "maximum": 180,
                "allow_null": True,
            },
            "description": (
                "Non-null start_lng values must be between "
                "-180 and 180."
            ),
        },
        "DQ017_END_LNG_RANGE": {
            "category": "VALIDITY",
            "columns": ["end_lng"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -180,
                "maximum": 180,
                "allow_null": True,
            },
            "description": (
                "Non-null end_lng values must be between "
                "-180 and 180."
            ),
        },

        # Coordinate completeness is separate from validity.
        "DQ018_START_LAT_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["start_lat"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing start latitude requires review."
            ),
        },
        "DQ019_END_LAT_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["end_lat"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing end latitude requires review."
            ),
        },
        "DQ020_START_LNG_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["start_lng"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing start longitude requires review."
            ),
        },
        "DQ021_END_LNG_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["end_lng"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing end longitude requires review."
            ),
        },
        "DQ022_STARTED_AT_PARSEABLE": {
            "category": "VALIDITY",
            "columns": ["started_at"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "DATETIME_PARSEABLE",
            "parameters": {},
            "description": (
                "Non-null started_at values must be valid "
                "datetimes."
            ),
        },
        "DQ023_ENDED_AT_PARSEABLE": {
            "category": "VALIDITY",
            "columns": ["ended_at"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "DATETIME_PARSEABLE",
            "parameters": {},
            "description": (
                "Non-null ended_at values must be valid "
                "datetimes."
            ),
        },
    },
    "run_history": [],
}


QUALITY_RULE_SECTION_TYPES = {
    "metadata": dict,
    "rules": dict,
    "run_history": list,
}


REQUIRED_RULE_FIELDS = {
    "category": str,
    "columns": list,
    "severity": str,
    "enabled": bool,
    "rule_type": str,
    "parameters": dict,
    "description": str,
}


SUPPORTED_QUALITY_SEVERITIES = {
    "WARNING",
    "FAIL",
}


SUPPORTED_QUALITY_RULE_TYPES = {
    "NOT_NULL",
    "NOT_BLANK",
    "UNIQUE",
    "ALLOWED_VALUES",
    "DATETIME_PARSEABLE",
    "DATETIME_ORDER",
    "DURATION_MAXIMUM",
    "NUMERIC_RANGE",
}


COORDINATE_RANGE_RULE_IDS = {
    "DQ014_START_LAT_RANGE",
    "DQ015_END_LAT_RANGE",
    "DQ016_START_LNG_RANGE",
    "DQ017_END_LNG_RANGE",
}


# ------------------------------------------
# Create and load quality rules
# ------------------------------------------

def create_quality_rules_if_missing(
    file_path: Path,
) -> bool:
    """
    Create the quality-rule file only when it is missing.
    """
    if file_path.exists():

        if not file_path.is_file():
            raise FileExistsError(
                "QUALITY_RULES_PATH exists but is not a file:\n"
                f"{file_path}"
            )

        log_quality_event(
            message=(
                "QUALITY_RULES_FOUND: Existing quality rules "
                "will be preserved and migrated where safe."
            ),
            level="INFO",
        )

        return False

    save_quality_json(
        file_path=file_path,
        data=deepcopy(DEFAULT_QUALITY_RULES),
    )

    log_quality_event(
        message=(
            "QUALITY_RULES_CREATED: quality_rules.json was "
            "created using ruleset version 1.1.0."
        ),
        level="INFO",
    )

    return True


def load_quality_rules(
    file_path: Path,
) -> tuple[dict, bool]:
    """
    Load quality rules while preserving malformed source files.
    """
    try:
        with file_path.open(
            mode="r",
            encoding="utf-8",
        ) as rules_file:
            loaded_rules = json.load(rules_file)

    except json.JSONDecodeError as error:
        log_quality_event(
            message=(
                "QUALITY_RULES_INVALID_JSON: "
                "Existing file preserved; in-memory defaults used. "
                f"Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_QUALITY_RULES), False

    except OSError as error:
        log_quality_event(
            message=(
                "QUALITY_RULES_READ_FAILED: "
                f"Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_QUALITY_RULES), False

    if not isinstance(loaded_rules, dict):
        log_quality_event(
            message=(
                "QUALITY_RULES_INVALID_STRUCTURE: "
                "The file must contain a JSON object."
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_QUALITY_RULES), False

    return loaded_rules, True


# ------------------------------------------
# Validate and migrate quality-rule registry
# ------------------------------------------

def validate_quality_rule_structure(
    rules_data: dict,
) -> tuple[dict, list[str], list[str]]:
    """
    Add missing top-level sections and identify invalid sections.
    """
    validated_rules = deepcopy(rules_data)

    missing_sections = []
    invalid_sections = []

    for section_name, expected_type in (
        QUALITY_RULE_SECTION_TYPES.items()
    ):
        if section_name not in validated_rules:
            validated_rules[section_name] = deepcopy(
                DEFAULT_QUALITY_RULES[section_name]
            )
            missing_sections.append(section_name)
            continue

        if not isinstance(
            validated_rules[section_name],
            expected_type,
        ):
            validated_rules[section_name] = deepcopy(
                DEFAULT_QUALITY_RULES[section_name]
            )
            invalid_sections.append(section_name)

    return validated_rules, missing_sections, invalid_sections


def migrate_quality_rules(
    rules_data: dict,
) -> list[str]:
    """
    Add new default rules and migrate the previous coordinate
    validity rules so null coordinates are assessed separately.
    """
    migration_changes = []

    configured_rules = rules_data["rules"]

    for rule_id, default_rule in (
        DEFAULT_QUALITY_RULES["rules"].items()
    ):
        if rule_id not in configured_rules:
            configured_rules[rule_id] = deepcopy(
                default_rule
            )
            migration_changes.append(
                f"ADDED_RULE:{rule_id}"
            )

    for rule_id in COORDINATE_RANGE_RULE_IDS:

        existing_rule = configured_rules.get(rule_id)

        if not isinstance(existing_rule, dict):
            continue

        parameters = existing_rule.get("parameters")

        if not isinstance(parameters, dict):
            continue

        if parameters.get("allow_null") is not True:
            parameters["allow_null"] = True
            migration_changes.append(
                f"ALLOW_NULL_ENABLED:{rule_id}"
            )

        default_description = (
            DEFAULT_QUALITY_RULES["rules"][
                rule_id
            ]["description"]
        )

        if (
            existing_rule.get("description")
            != default_description
        ):
            existing_rule[
                "description"
            ] = default_description

            migration_changes.append(
                f"DESCRIPTION_UPDATED:{rule_id}"
            )

    metadata = rules_data["metadata"]

    metadata_defaults = DEFAULT_QUALITY_RULES[
        "metadata"
    ]

    for field_name, default_value in (
        metadata_defaults.items()
    ):
        if field_name not in metadata:
            metadata[field_name] = deepcopy(
                default_value
            )
            migration_changes.append(
                f"ADDED_METADATA:{field_name}"
            )

    if metadata.get("ruleset_version") != "1.1.0":
        metadata["ruleset_version"] = "1.1.0"
        migration_changes.append(
            "RULESET_VERSION_UPDATED:1.1.0"
        )

    if (
        metadata.get("source_schema_version")
        != ACTIVE_SCHEMA_VERSION
    ):
        metadata[
            "source_schema_version"
        ] = ACTIVE_SCHEMA_VERSION

        migration_changes.append(
            "SOURCE_SCHEMA_VERSION_UPDATED:"
            f"{ACTIVE_SCHEMA_VERSION}"
        )

    if migration_changes:
        metadata["updated_at"] = datetime.now(
            MELBOURNE_TIMEZONE
        ).isoformat(timespec="seconds")

    return migration_changes


def validate_quality_rule_definitions(
    rules_data: dict,
    approved_schema: dict,
) -> list[dict]:
    """
    Validate rule structure and approved-schema references.
    """
    validation_issues = []

    for rule_id, rule_definition in (
        rules_data["rules"].items()
    ):
        if (
            not isinstance(rule_id, str)
            or not rule_id.strip()
        ):
            validation_issues.append(
                {
                    "rule_id": str(rule_id),
                    "issue": "INVALID_RULE_ID",
                }
            )
            continue

        if not isinstance(rule_definition, dict):
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": "RULE_NOT_OBJECT",
                }
            )
            continue

        for field_name, expected_type in (
            REQUIRED_RULE_FIELDS.items()
        ):
            if field_name not in rule_definition:
                validation_issues.append(
                    {
                        "rule_id": rule_id,
                        "issue": "MISSING_FIELD",
                        "field": field_name,
                    }
                )
                continue

            if not isinstance(
                rule_definition[field_name],
                expected_type,
            ):
                validation_issues.append(
                    {
                        "rule_id": rule_id,
                        "issue": "INVALID_FIELD_TYPE",
                        "field": field_name,
                    }
                )

        severity = rule_definition.get("severity")

        if severity not in SUPPORTED_QUALITY_SEVERITIES:
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": "UNSUPPORTED_SEVERITY",
                    "observed": severity,
                }
            )

        rule_type = rule_definition.get("rule_type")

        if rule_type not in SUPPORTED_QUALITY_RULE_TYPES:
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": "UNSUPPORTED_RULE_TYPE",
                    "observed": rule_type,
                }
            )

        if rule_definition.get("enabled") is not True:
            continue

        rule_columns = rule_definition.get(
            "columns",
            [],
        )

        if not isinstance(rule_columns, list):
            continue

        missing_schema_columns = [
            column_name
            for column_name in rule_columns
            if column_name not in approved_schema
        ]

        if missing_schema_columns:
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": (
                        "COLUMN_NOT_IN_APPROVED_SCHEMA"
                    ),
                    "columns": missing_schema_columns,
                }
            )

    return validation_issues


# ------------------------------------------
# Initialise and migrate quality rules
# ------------------------------------------

quality_rules_created = (
    create_quality_rules_if_missing(
        file_path=QUALITY_RULES_PATH,
    )
)

(
    quality_rules_raw,
    quality_rules_loaded_from_disk,
) = load_quality_rules(
    file_path=QUALITY_RULES_PATH,
)


(
    quality_rules,
    quality_rules_missing_sections,
    quality_rules_invalid_sections,
) = validate_quality_rule_structure(
    rules_data=quality_rules_raw,
)


quality_rule_migration_changes = []

if (
    "rules" not in quality_rules_invalid_sections
    and "metadata"
    not in quality_rules_invalid_sections
):
    quality_rule_migration_changes = (
        migrate_quality_rules(
            rules_data=quality_rules,
        )
    )


quality_rule_validation_issues = (
    validate_quality_rule_definitions(
        rules_data=quality_rules,
        approved_schema=ACTIVE_SCHEMA,
    )
)


safe_quality_rule_migration = (
    quality_rules_loaded_from_disk
    and bool(quality_rule_migration_changes)
    and not quality_rules_invalid_sections
    and not quality_rule_validation_issues
)


if safe_quality_rule_migration:
    save_quality_json(
        file_path=QUALITY_RULES_PATH,
        data=quality_rules,
    )

    log_quality_event(
        message=(
            "QUALITY_RULES_MIGRATION_SAVED: "
            f"changes={len(quality_rule_migration_changes)}."
        ),
        level="WARNING",
    )


for issue in quality_rule_validation_issues:
    log_quality_event(
        message=(
            "QUALITY_RULE_VALIDATION_ISSUE: "
            f"{issue}"
        ),
        level="ERROR",
    )


QUALITY_RULES_READY = (
    not quality_rules_invalid_sections
    and not quality_rule_validation_issues
)


ENABLED_QUALITY_RULES = {
    rule_id: rule_definition
    for rule_id, rule_definition in (
        quality_rules["rules"].items()
    )
    if rule_definition.get("enabled") is True
}


if QUALITY_RULES_READY:
    log_quality_event(
        message=(
            "QUALITY_RULES_READY: "
            f"rules={len(ENABLED_QUALITY_RULES)}; "
            f"ruleset_version="
            f"{quality_rules['metadata']['ruleset_version']}; "
            f"schema_version={ACTIVE_SCHEMA_VERSION}."
        ),
        level="INFO",
    )


# ------------------------------------------
# Display Part 3.1 summary
# ------------------------------------------

failure_rule_count = sum(
    1
    for rule in ENABLED_QUALITY_RULES.values()
    if rule.get("severity") == "FAIL"
)

warning_rule_count = sum(
    1
    for rule in ENABLED_QUALITY_RULES.values()
    if rule.get("severity") == "WARNING"
)


print(
    "\n========== Data Quality Rule Initialisation "
    "==========\n"
)

print(f"Quality audit session ID : {QUALITY_AUDIT_SESSION_ID}")
print(f"Quality rules path       : {QUALITY_RULES_PATH}")
print(f"Quality rules created    : {quality_rules_created}")
print(
    f"Ruleset version          : "
    f"{quality_rules['metadata'].get('ruleset_version')}"
)
print(
    f"Source schema version    : "
    f"{quality_rules['metadata'].get('source_schema_version')}"
)
print(f"Enabled rules            : {len(ENABLED_QUALITY_RULES)}")
print(f"Failure rules            : {failure_rule_count}")
print(f"Warning rules            : {warning_rule_count}")
print(
    f"Migration changes        : "
    f"{len(quality_rule_migration_changes)}"
)
print(
    f"Validation issues        : "
    f"{len(quality_rule_validation_issues)}"
)

print("\n" + "=" * 70)

if QUALITY_RULES_READY:
    print("✓ Approved active schema verified.")
    print("✓ Quality rules created, loaded, or migrated.")
    print("✓ Coordinate completeness and validity are separate.")
    print("✓ Rules are ready for Part 3.2.")
else:
    print("✗ Quality rules require correction.")

print("=" * 70)

## Part 3.2: Generate File-Level and Dataset-Level Quality Metrics

### Objective

Scan every discovered CSV file and generate the measurements required for the data-quality assessment without loading the complete dataset into memory.

This section collects evidence only. It does not determine whether a quality rule passes, warns, or fails. Rule assessment will occur in Part 3.3.

### Full-File Scanning Policy

Unlike schema profiling, data-quality auditing must inspect the complete source dataset.

Sample-based scanning is not sufficient for reliably detecting:

* duplicate ride IDs;
* missing values;
* invalid category values;
* datetime inconsistencies;
* duration anomalies;
* coordinate violations;
* row-level quarantine candidates.

Each CSV is therefore read completely in configurable chunks.

The default chunk size is:

```text
100,000 rows
```

Only one chunk is held in memory at a time.

### Memory-Control Strategy

The audit keeps compact information in memory:

* row and column counts;
* null and blank counts;
* category frequencies;
* datetime and duration metrics;
* coordinate metrics;
* station-field metrics;
* bounded examples of affected rows;
* file-level and dataset-level summaries.

The audit does not retain:

* complete DataFrames;
* one combined DataFrame;
* every affected row as a Python object;
* every ride ID in memory.

### Duplicate Tracking

Duplicate `ride_id` detection must operate across chunks and files.

A temporary SQLite database is used to store:

```text
ride_id
source_file
source_row_number
```

After all files have been scanned, SQLite is queried to identify:

* duplicates within an individual file;
* duplicates across multiple files;
* total duplicate ride IDs;
* total rows affected by duplication.

The temporary database is removed after the required metrics and bounded evidence have been extracted.

### Source Lineage

Evidence records contain:

* source filename;
* source row number;
* ride ID when available;
* observed value or relevant context.

The evidence collection is bounded to prevent common issues from generating millions of in-memory records.

The default evidence limit is:

```text
100 examples per metric
```

Counts always represent the complete scan, even when only a limited number of examples are retained.

### Metrics Generated

For each file, this section measures:

* rows scanned;
* columns available;
* missing approved-schema columns;
* null counts by column;
* blank-string counts by column;
* observed `rideable_type` values;
* observed `member_casual` values;
* invalid or unparseable timestamps;
* non-positive ride durations;
* rides longer than 24 hours;
* minimum, maximum, and average ride duration;
* missing station names and IDs;
* non-numeric latitude and longitude values;
* coordinates below or above global geographic ranges;
* duplicate ride IDs within the file.

Dataset-level metrics include:

* total rows scanned;
* total files scanned;
* aggregate null and blank counts;
* combined category frequencies;
* aggregate datetime, duration, station, and coordinate metrics;
* duplicate ride IDs across the complete dataset.

### Failure Handling

Each file is processed independently.

If one file cannot be scanned:

* the failure is recorded;
* scanning continues with the remaining files;
* partial metrics are marked clearly;
* the file failure is not automatically classified as a quality-rule failure.

### In-Memory Outputs

This section produces:

```text
quality_metrics_by_file
dataset_quality_metrics
quality_metric_evidence
quality_scan_failures
```

These results remain in memory.

Part 3.3 will compare them with `quality_rules.json` and assign rule outcomes.

### Expected Outcome

* All discoverable source files are scanned in chunks.
* Full-dataset quality metrics are generated without combining all data in memory.
* Cross-file duplicate ride IDs are identified through disk-backed tracking.
* Evidence samples remain bounded.
* File-processing failures are recorded separately.
* Metrics are ready for quality-rule assessment in Part 3.3.


In [ ]:
# ==========================================
# Part 3.3: Execute Data Quality Assessment
# ==========================================

from collections import Counter
from copy import deepcopy
from datetime import datetime


# ------------------------------------------
# Validate Part 3.3 dependencies
# ------------------------------------------

if not QUALITY_RULES_READY:
    raise RuntimeError(
        "Part 3.3 cannot continue because the quality rules "
        "are not ready."
    )

if not isinstance(
    quality_metrics_by_file,
    dict,
):
    raise TypeError(
        "quality_metrics_by_file must be a dictionary."
    )

if not isinstance(
    dataset_quality_metrics,
    dict,
):
    raise TypeError(
        "dataset_quality_metrics must be a dictionary."
    )


# ------------------------------------------
# Initialise assessment outputs
# ------------------------------------------

quality_assessment_by_file = {}
dataset_quality_assessment = {}

quality_rule_results = []
quarantine_rule_candidates = []

quality_assessment_start_time = (
    datetime.now(MELBOURNE_TIMEZONE)
)


# ------------------------------------------
# Assessment helpers
# ------------------------------------------

def determine_quality_outcome(
    severity: str,
    affected_count: int,
) -> str:
    """
    Assign PASS, WARNING, or FAIL from severity and affected count.
    """
    if affected_count <= 0:
        return "PASS"

    if severity == "WARNING":
        return "WARNING"

    if severity == "FAIL":
        return "FAIL"

    raise ValueError(
        f"Unsupported severity: {severity}"
    )


def get_metric_evidence(
    metric_name: str,
    file_name: str | None = None,
) -> list[dict]:
    """
    Retrieve bounded evidence, optionally filtered by filename.
    """
    evidence_records = (
        quality_metric_evidence.get(
            metric_name,
            [],
        )
    )

    if file_name is None:
        return deepcopy(
            evidence_records
        )

    return [
        deepcopy(record)
        for record in evidence_records
        if record.get("file_name")
        == file_name
    ]


def combine_metric_evidence(
    metric_names: list[str],
    file_name: str | None = None,
) -> list[dict]:
    """
    Combine bounded evidence from related metric categories.
    """
    combined_records = []

    for metric_name in metric_names:
        combined_records.extend(
            get_metric_evidence(
                metric_name=metric_name,
                file_name=file_name,
            )
        )

    return combined_records[
        :DQ_EVIDENCE_LIMIT
    ]


def build_rule_result(
    rule_id: str,
    rule_definition: dict,
    scope: str,
    affected_count: int,
    evaluated_rows: int,
    reason: str,
    file_name: str | None = None,
    observed_details: dict | None = None,
    evidence: list[dict] | None = None,
) -> dict:
    """
    Build a standard rule-assessment record.
    """
    outcome = determine_quality_outcome(
        severity=rule_definition["severity"],
        affected_count=affected_count,
    )

    result = {
        "rule_id": rule_id,
        "category": (
            rule_definition["category"]
        ),
        "rule_type": (
            rule_definition["rule_type"]
        ),
        "severity": (
            rule_definition["severity"]
        ),
        "columns": deepcopy(
            rule_definition["columns"]
        ),
        "scope": scope,
        "file_name": file_name,
        "evaluated_rows": int(
            evaluated_rows
        ),
        "affected_count": int(
            affected_count
        ),
        "outcome": outcome,
        "reason": reason,
        "description": (
            rule_definition["description"]
        ),
        "observed_details": deepcopy(
            observed_details or {}
        ),
        "evidence": deepcopy(
            evidence or []
        ),
    }

    if outcome == "FAIL":
        quarantine_rule_candidates.append(
            {
                "rule_id": rule_id,
                "file_name": file_name,
                "scope": scope,
                "affected_count": int(
                    affected_count
                ),
                "reason": reason,
                "evidence": deepcopy(
                    result["evidence"]
                ),
            }
        )

    return result


def evaluate_allowed_values(
    observed_counts: dict,
    allowed_values: list,
    case_sensitive: bool,
) -> tuple[int, dict]:
    """
    Count category values outside the approved set.
    """
    if case_sensitive:
        allowed_set = set(
            allowed_values
        )

        invalid_counts = {
            value: count
            for value, count
            in observed_counts.items()
            if value not in allowed_set
        }

    else:
        allowed_set = {
            str(value).casefold()
            for value in allowed_values
        }

        invalid_counts = {
            value: count
            for value, count
            in observed_counts.items()
            if str(value).casefold()
            not in allowed_set
        }

    return (
        sum(invalid_counts.values()),
        invalid_counts,
    )


# ------------------------------------------
# Evaluate one file-level rule
# ------------------------------------------

def evaluate_file_rule(
    rule_id: str,
    rule_definition: dict,
    file_name: str,
    file_metrics: dict,
) -> dict:
    """
    Evaluate one rule against one successfully scanned file.
    """
    rule_type = (
        rule_definition["rule_type"]
    )

    rule_columns = (
        rule_definition["columns"]
    )

    parameters = (
        rule_definition["parameters"]
    )

    evaluated_rows = file_metrics[
        "rows_scanned"
    ]

    missing_columns = [
        column_name
        for column_name in rule_columns
        if column_name
        in file_metrics.get(
            "missing_quality_columns",
            [],
        )
    ]

    if missing_columns:
        return build_rule_result(
            rule_id=rule_id,
            rule_definition=rule_definition,
            scope="FILE",
            file_name=file_name,
            affected_count=max(
                evaluated_rows,
                1,
            ),
            evaluated_rows=evaluated_rows,
            reason="COLUMN_MISSING",
            observed_details={
                "missing_columns": (
                    missing_columns
                ),
            },
            evidence=get_metric_evidence(
                metric_name=(
                    "missing_quality_columns"
                ),
                file_name=file_name,
            ),
        )


    # --------------------------------------
    # NOT_NULL
    # --------------------------------------

    if rule_type == "NOT_NULL":
        column_name = rule_columns[0]

        if column_name in STATION_COLUMNS:
            affected_count = (
                file_metrics[
                    "station_missing_counts"
                ].get(column_name, 0)
            )

            evidence_metric = (
                f"missing_{column_name}"
            )

        else:
            affected_count = (
                file_metrics[
                    "null_counts"
                ].get(column_name, 0)
            )

            evidence_metric = (
                f"null_{column_name}"
            )

        return build_rule_result(
            rule_id=rule_id,
            rule_definition=rule_definition,
            scope="FILE",
            file_name=file_name,
            affected_count=affected_count,
            evaluated_rows=evaluated_rows,
            reason="NULL_OR_MISSING_VALUE",
            observed_details={
                "column": column_name,
            },
            evidence=get_metric_evidence(
                metric_name=evidence_metric,
                file_name=file_name,
            ),
        )


    # --------------------------------------
    # NOT_BLANK
    # --------------------------------------

    if rule_type == "NOT_BLANK":
        column_name = rule_columns[0]

        affected_count = (
            file_metrics[
                "blank_counts"
            ].get(column_name, 0)
        )

        return build_rule_result(
            rule_id=rule_id,
            rule_definition=rule_definition,
            scope="FILE",
            file_name=file_name,
            affected_count=affected_count,
            evaluated_rows=evaluated_rows,
            reason="BLANK_STRING",
            observed_details={
                "column": column_name,
            },
            evidence=get_metric_evidence(
                metric_name=(
                    f"blank_{column_name}"
                ),
                file_name=file_name,
            ),
        )


    # --------------------------------------
    # ALLOWED_VALUES
    # --------------------------------------

    if rule_type == "ALLOWED_VALUES":
        column_name = rule_columns[0]

        observed_counts = (
            file_metrics[
                "category_counts"
            ].get(column_name, {})
        )

        (
            affected_count,
            invalid_counts,
        ) = evaluate_allowed_values(
            observed_counts=observed_counts,
            allowed_values=parameters[
                "allowed_values"
            ],
            case_sensitive=parameters.get(
                "case_sensitive",
                True,
            ),
        )

        return build_rule_result(
            rule_id=rule_id,
            rule_definition=rule_definition,
            scope="FILE",
            file_name=file_name,
            affected_count=affected_count,
            evaluated_rows=evaluated_rows,
            reason="VALUE_NOT_ALLOWED",
            observed_details={
                "allowed_values": deepcopy(
                    parameters[
                        "allowed_values"
                    ]
                ),
                "invalid_value_counts": (
                    invalid_counts
                ),
            },
            evidence=get_metric_evidence(
                metric_name=(
                    f"invalid_{column_name}"
                ),
                file_name=file_name,
            ),
        )


    # --------------------------------------
    # DATETIME_PARSEABLE
    # --------------------------------------

    if rule_type == "DATETIME_PARSEABLE":
        column_name = rule_columns[0]

        metric_name = (
            f"{column_name}_unparseable"
        )

        affected_count = (
            file_metrics[
                "datetime_metrics"
            ].get(metric_name, 0)
        )

        return build_rule_result(
            rule_id=rule_id,
            rule_definition=rule_definition,
            scope="FILE",
            file_name=file_name,
            affected_count=affected_count,
            evaluated_rows=evaluated_rows,
            reason="DATETIME_UNPARSEABLE",
            observed_details={
                "column": column_name,
            },
            evidence=get_metric_evidence(
                metric_name=(
                    f"unparseable_"
                    f"{column_name}"
                ),
                file_name=file_name,
            ),
        )


    # --------------------------------------
    # DATETIME_ORDER
    # --------------------------------------

    if rule_type == "DATETIME_ORDER":
        affected_count = (
            file_metrics[
                "datetime_metrics"
            ][
                "non_positive_duration"
            ]
        )

        return build_rule_result(
            rule_id=rule_id,
            rule_definition=rule_definition,
            scope="FILE",
            file_name=file_name,
            affected_count=affected_count,
            evaluated_rows=evaluated_rows,
            reason="DATETIME_ORDER_VIOLATION",
            evidence=get_metric_evidence(
                metric_name=(
                    "non_positive_duration"
                ),
                file_name=file_name,
            ),
        )


    # --------------------------------------
    # DURATION_MAXIMUM
    # --------------------------------------

    if rule_type == "DURATION_MAXIMUM":
        affected_count = (
            file_metrics[
                "datetime_metrics"
            ][
                "duration_over_maximum"
            ]
        )

        return build_rule_result(
            rule_id=rule_id,
            rule_definition=rule_definition,
            scope="FILE",
            file_name=file_name,
            affected_count=affected_count,
            evaluated_rows=evaluated_rows,
            reason="DURATION_ABOVE_MAXIMUM",
            observed_details={
                "maximum_hours": (
                    parameters[
                        "maximum_hours"
                    ]
                ),
            },
            evidence=get_metric_evidence(
                metric_name=(
                    "duration_over_maximum"
                ),
                file_name=file_name,
            ),
        )


    # --------------------------------------
    # NUMERIC_RANGE
    # --------------------------------------

    if rule_type == "NUMERIC_RANGE":
        column_name = rule_columns[0]

        coordinate_metrics = (
            file_metrics[
                "coordinate_metrics"
            ][column_name]
        )

        affected_count = (
            coordinate_metrics[
                "non_numeric_count"
            ]
            + coordinate_metrics[
                "below_minimum_count"
            ]
            + coordinate_metrics[
                "above_maximum_count"
            ]
        )

        allow_null = parameters.get(
            "allow_null",
            True,
        )

        evidence_metrics = [
            f"coordinate_non_numeric_"
            f"{column_name}",
            f"coordinate_below_minimum_"
            f"{column_name}",
            f"coordinate_above_maximum_"
            f"{column_name}",
        ]

        if not allow_null:
            affected_count += (
                coordinate_metrics[
                    "null_count"
                ]
            )

            evidence_metrics.append(
                f"coordinate_null_"
                f"{column_name}"
            )

        return build_rule_result(
            rule_id=rule_id,
            rule_definition=rule_definition,
            scope="FILE",
            file_name=file_name,
            affected_count=affected_count,
            evaluated_rows=evaluated_rows,
            reason="NUMERIC_RANGE_VIOLATION",
            observed_details={
                "minimum": parameters[
                    "minimum"
                ],
                "maximum": parameters[
                    "maximum"
                ],
                "allow_null": allow_null,
                "metric_counts": deepcopy(
                    coordinate_metrics
                ),
            },
            evidence=combine_metric_evidence(
                metric_names=(
                    evidence_metrics
                ),
                file_name=file_name,
            ),
        )


    raise ValueError(
        f"Unsupported file-level rule type: "
        f"{rule_type}"
    )


# ------------------------------------------
# Evaluate dataset-level uniqueness
# ------------------------------------------

def evaluate_dataset_unique_rule(
    rule_id: str,
    rule_definition: dict,
) -> dict:
    """
    Evaluate duplicate ride IDs across all scanned files.
    """
    duplicate_metrics = (
        dataset_quality_metrics[
            "duplicates"
        ]
    )

    affected_count = (
        duplicate_metrics[
            "duplicate_rows_affected"
        ]
    )

    return build_rule_result(
        rule_id=rule_id,
        rule_definition=rule_definition,
        scope="DATASET",
        file_name=None,
        affected_count=affected_count,
        evaluated_rows=(
            dataset_quality_metrics[
                "total_rows_scanned"
            ]
        ),
        reason="DUPLICATE_VALUE",
        observed_details=deepcopy(
            duplicate_metrics
        ),
        evidence=get_metric_evidence(
            metric_name=(
                "duplicate_ride_id"
            ),
        ),
    )


# ------------------------------------------
# Execute assessment
# ------------------------------------------

log_quality_event(
    message=(
        "DATA_QUALITY_ASSESSMENT_START: "
        f"rules={len(ENABLED_QUALITY_RULES)}; "
        f"files={len(quality_metrics_by_file)}."
    ),
    level="INFO",
)


for rule_id, rule_definition in (
    ENABLED_QUALITY_RULES.items()
):
    rule_type = (
        rule_definition["rule_type"]
    )

    rule_scope = (
        rule_definition.get(
            "parameters",
            {},
        ).get(
            "scope",
            "FILE",
        )
    )

    if (
        rule_type == "UNIQUE"
        and rule_scope == "ALL_FILES"
    ):
        dataset_result = (
            evaluate_dataset_unique_rule(
                rule_id=rule_id,
                rule_definition=(
                    rule_definition
                ),
            )
        )

        dataset_quality_assessment[
            rule_id
        ] = dataset_result

        quality_rule_results.append(
            dataset_result
        )

        continue

    for file_name, file_metrics in (
        quality_metrics_by_file.items()
    ):
        if (
            file_metrics.get("status")
            != "COMPLETE"
        ):
            continue

        file_result = evaluate_file_rule(
            rule_id=rule_id,
            rule_definition=(
                rule_definition
            ),
            file_name=file_name,
            file_metrics=file_metrics,
        )

        quality_assessment_by_file.setdefault(
            file_name,
            {},
        )[rule_id] = file_result

        quality_rule_results.append(
            file_result
        )


# ------------------------------------------
# Build summary
# ------------------------------------------

outcome_counts = Counter(
    result["outcome"]
    for result in quality_rule_results
)


file_outcome_summary = {}

for file_name, file_results in (
    quality_assessment_by_file.items()
):
    file_counts = Counter(
        result["outcome"]
        for result in file_results.values()
    )

    if file_counts["FAIL"] > 0:
        overall_file_outcome = "FAIL"

    elif file_counts["WARNING"] > 0:
        overall_file_outcome = "WARNING"

    else:
        overall_file_outcome = "PASS"

    file_outcome_summary[file_name] = {
        "overall_outcome": (
            overall_file_outcome
        ),
        "pass_count": (
            file_counts["PASS"]
        ),
        "warning_count": (
            file_counts["WARNING"]
        ),
        "fail_count": (
            file_counts["FAIL"]
        ),
    }


if outcome_counts["FAIL"] > 0:
    overall_quality_outcome = "FAIL"

elif outcome_counts["WARNING"] > 0:
    overall_quality_outcome = "WARNING"

else:
    overall_quality_outcome = "PASS"


assessment_execution_status = (
    "PARTIAL_FAILURE"
    if quality_scan_failures
    else "COMPLETE"
)


quality_assessment_end_time = (
    datetime.now(MELBOURNE_TIMEZONE)
)


quality_assessment_summary = {
    "quality_audit_session_id": (
        QUALITY_AUDIT_SESSION_ID
    ),
    "started_at": (
        quality_assessment_start_time
        .isoformat(timespec="seconds")
    ),
    "completed_at": (
        quality_assessment_end_time
        .isoformat(timespec="seconds")
    ),
    "assessment_execution_status": (
        assessment_execution_status
    ),
    "overall_quality_outcome": (
        overall_quality_outcome
    ),
    "enabled_rules": len(
        ENABLED_QUALITY_RULES
    ),
    "rule_evaluations": len(
        quality_rule_results
    ),
    "pass_count": outcome_counts["PASS"],
    "warning_count": (
        outcome_counts["WARNING"]
    ),
    "fail_count": outcome_counts["FAIL"],
    "files_assessed": len(
        quality_assessment_by_file
    ),
    "files_not_assessed": len(
        quality_scan_failures
    ),
    "quarantine_rule_candidate_count": len(
        quarantine_rule_candidates
    ),
    "file_outcomes": (
        file_outcome_summary
    ),
}


log_quality_event(
    message=(
        "DATA_QUALITY_ASSESSMENT_COMPLETE: "
        f"execution_status="
        f"{assessment_execution_status}; "
        f"quality_outcome="
        f"{overall_quality_outcome}; "
        f"pass={outcome_counts['PASS']}; "
        f"warning="
        f"{outcome_counts['WARNING']}; "
        f"fail={outcome_counts['FAIL']}."
    ),
    level="INFO",
)


# ------------------------------------------
# Display Part 3.3 summary
# ------------------------------------------

print(
    "\n========== Data Quality Assessment "
    "==========\n"
)

print(
    f"Quality audit session ID  : "
    f"{QUALITY_AUDIT_SESSION_ID}"
)
print(
    f"Execution status          : "
    f"{assessment_execution_status}"
)
print(
    f"Overall quality outcome   : "
    f"{overall_quality_outcome}"
)
print(
    f"Enabled rules             : "
    f"{len(ENABLED_QUALITY_RULES)}"
)
print(
    f"Rule evaluations          : "
    f"{len(quality_rule_results)}"
)
print(
    f"Passed evaluations        : "
    f"{outcome_counts['PASS']}"
)
print(
    f"Warning evaluations       : "
    f"{outcome_counts['WARNING']}"
)
print(
    f"Failed evaluations        : "
    f"{outcome_counts['FAIL']}"
)
print(
    f"Quarantine candidates     : "
    f"{len(quarantine_rule_candidates)}"
)


print("\nFile outcomes:")

for file_name, file_summary in (
    file_outcome_summary.items()
):
    print(
        f"  {file_name:<40} "
        f"{file_summary['overall_outcome']:<8} "
        f"pass={file_summary['pass_count']:<3} "
        f"warning="
        f"{file_summary['warning_count']:<3} "
        f"fail={file_summary['fail_count']:<3}"
    )


print("\n" + "=" * 75)

remaining_failures = [
    result
    for result in quality_rule_results
    if result["outcome"] == "FAIL"
]

for result in remaining_failures:
    print(f"Rule ID        : {result['rule_id']}")
    print(f"File           : {result['file_name'] or 'ALL_FILES'}")
    print(f"Affected rows  : {result['affected_count']:,}")
    print(f"Reason         : {result['reason']}")
    print(f"Evidence rows  : {len(result['evidence'])}")
    print("-" * 70)

if overall_quality_outcome == "PASS":
    print("✓ All evaluated quality rules passed.")

elif overall_quality_outcome == "WARNING":
    print(
        "⚠ No failure rules were violated, but review is required."
    )

else:
    print(
        "✗ One or more failure-severity rules were violated."
    )

print("✓ Results are ready for Part 3.4.")
print("=" * 75)